# Forge Neo on Google Colab - SDXL

Generated from the **verified source** of `https://github.com/Haoming02/sd-webui-forge-classic`, branch `neo`
(`modules/launch_utils.py`, `modules/cmd_args.py`, `modules/paths_internal.py`,
`requirements.txt`). Where the README and the code disagree, the code wins.

### Verified pins
* **`neo` is the repo's default branch.** Last commit 2026-10-03 (Classic: 2026-04-22).
* Python **3.13** (tested 3.13.12).
* Gradio **4.40.0** + `gradio_rangeslider==0.0.8`.
* torch `2.13.0+cu130` / torchvision `0.28.0+cu130`.
* xformers `0.0.35` requires **torch>=2.10** (looser than Classic).
* cu128 has **no** torch 2.13.0 cp313 build - the fallback is torch **2.11.0**.
* `requirements_versions.txt` does not exist (HTTP 404).
* Adds model families Classic lacks: qwen, flux1, flux2, PiD 1.5, Nunchaku SVDQ, Wan.

### How to use
Run the cells **top to bottom, in order**. Cell 1 is the only one you normally edit.
Every later cell depends on the ones before it, so if one fails, fix it and re-run
from that point - later cells validate what the earlier ones produced.

**Runtime**: *Runtime -> Change runtime type -> GPU (T4 / L4)*.
> ⚠️ T4 has **15 GB VRAM**. Turn Hires.fix off, keep the text encoder on CPU.

---
## Cell 1 — Configuration

The only cell you normally need to edit. Everything downstream reads `CFG`.

In [ ]:
# %% Cell 1 - CONFIGURATION
CFG = {
    "repo":        "https://github.com/Haoming02/sd-webui-forge-classic",
    "branch":      "neo",
    "install_dir": "/content/sd-webui-forge-neo",
    "drive_root":  "MyDrive/forge-neo",

    # Python the WebUI requires (Forge Neo is tested with 3.13)
    "pyver":       "3.13",

    # "auto" picks cu130 when the Colab driver supports CUDA 13, else cu128.
    # Force to "cu130" / "cu128" to override the auto-detection.
    "torch_build": "auto",
    # available builds:
    "builds": {
        "cu130": ("2.13.0", "0.28.0"),
        "cu128": ("2.11.0", "0.26.0"),
    },

    "gradio_package": "gradio==4.40.0 gradio_rangeslider==0.0.8",
    "gradio_expect":  "4.40.0",
    "xformers":       "0.0.35",
    "use_xformers":   True,

    "port": 7860,
    "install_adetailer": True,

    # Extra flags appended to COMMANDLINE_ARGS.
    # Tip: --skip-prepare-environment and --skip-install speed up later boots,
    # but REMOVE them before installing an extension.
    "extra_launch_args": ['--expandable-segments'],
}

print("CONFIG:")
for k, v in CFG.items():
    print(f"  {k:<20} {v}")

---
## Cell 2 — Preflight & error-aware runner

Installs nothing. Checks the GPU, the driver (to pick the right CUDA build), disk
space and Python, and defines `run()`, which streams output and raises a
`ForgeError` carrying a **targeted hint** instead of a raw traceback.

In [ ]:
# %% Cell 2 - PREFLIGHT + ERROR-AWARE RUNNER
import os, re, sys, json, shlex, shutil, subprocess, pathlib, collections

INSTALL = CFG["install_dir"]
DRIVE = "/content/drive"
ROOT = f"{DRIVE}/{CFG['drive_root']}"

# ---- error hint engine -------------------------------------------------------
HINTS = [
    (r"uv: command not found",
     "uv is not installed. Run the 'Install uv + clone' cell first."),
    (r"command not found: git",
     "git missing from this image. Install it, or restart the runtime."),
    (r"No space left on device",
     "Out of disk. Delete venv/ and re-run, or use --uv-symlink to cut ~7GB to ~100MB."),
    (r"driver version is insufficient for CUDA runtime|CUDA driver version is insufficient",
     "The Colab driver is older than the CUDA build. Set CFG['torch_build']='cu128'."),
    (r"Please update your GPU driver to support cu130",
     "cu130 needs driver >= 580. Set CFG['torch_build']='cu128'."),
    (r"Torch not compiled with CUDA",
     "A CPU/ROCm torch wheel was picked up. Delete venv/ and re-run the venv + build cells."),
    (r"RuntimeError.*compiled with different CUDA versions",
     "torch and torchvision builds disagree. Re-run the build cell so both are pinned together."),
    (r"undefined symbol|cannot open shared object file.*libcud|libcuda",
     "A native CUDA/xformers library mismatch. Re-run the build cell, or set use_xformers=False."),
    (r"No module named ['\"]?gradio",
     "gradio was not installed. Re-run the build cell (do not pass --skip-install the first time)."),
    (r"cannot import name .* from .*gradio|ImportError: cannot import name",
     "Wrong gradio major version. Check the verify cell and re-pin GRADIO_PACKAGE."),
    (r"This program is tested with 3\.\d+",
     "Wrong Python version. Colab ships 3.12 - the venv cell must fetch a standalone interpreter."),
    (r"disable_safe_unpickle",
     "ADetailer vs torch>=2.6. Ensure ENV['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='true'."),
    (r"SSLError|ProxyError|Could not resolve host|Temporary failure in name resolution",
     "Network hiccup. Re-run the cell; Colab egress is occasionally flaky."),
    (r"Directory not empty|already exists and is not an empty directory",
     "Leftover from a previous run. The clone/venv cell handles this by removing it first."),
]

def hint_for(text):
    for pat, h in HINTS:
        if re.search(pat, text, re.I):
            return h
    return None

class ForgeError(RuntimeError):
    def __init__(self, msg, tail, hint):
        self.tail, self.hint = tail, hint
        super().__init__(msg)

def _fmt(cmd):
    return cmd if isinstance(cmd, str) else " ".join(shlex.quote(c) for c in cmd)

def run(cmd, cwd=None, env=None, check=True, echo=True, tail=45):
    """Run a command, streaming output. On failure raise ForgeError with a hint."""
    argv = cmd if isinstance(cmd, str) else list(cmd)
    if echo:
        print(f"$ {_fmt(argv)}", flush=True)
    proc = subprocess.Popen(
        argv, cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    buf = collections.deque(maxlen=tail)
    for line in proc.stdout:
        if echo:
            print(line, end="", flush=True)
        buf.append(line)
    proc.wait()
    text = "".join(buf)
    if check and proc.returncode != 0:
        raise ForgeError(f"`{_fmt(argv)}` failed with exit code {proc.returncode}", text,
                         hint_for(text))
    return proc.returncode, text

def quiet(cmd, **kw):
    kw["echo"] = False
    return run(cmd, **kw)

def step(n, title):
    print(f"\n{'='*66}\n  STEP {n}: {title}\n{'='*66}", flush=True)

# ---- preflight --------------------------------------------------------------
info = {}
try:
    _, smi = quiet(["nvidia-smi",
                    "--query-gpu=name,memory.total,driver_version,compute_cap",
                    "--format=csv,noheader"], check=False)
    rows = [r.strip() for r in smi.strip().splitlines() if r.strip()]
    if rows:
        parts = [p.strip() for p in rows[0].split(",")]
        info["gpu"] = parts[0]
        info["vram_mb"] = int(re.sub(r"[^0-9]", "", parts[1]) or 0)
        drv = parts[2]
        info["driver"] = drv
        m = re.match(r"(\d+)\.(\d+)", drv)
        dmaj = int(m.group(1)) if m else 0
        info["driver_major"] = dmaj
        info["supports_cu130"] = dmaj >= 580
        if len(parts) > 3 and parts[3]:
            info["compute_cap"] = parts[3].replace(".", "").strip()
except Exception as e:
    print(f"nvidia-smi failed: {e}")

try:
    du = shutil.disk_usage("/content")
    info["disk_free_gb"] = round(du.free / 1e9, 1)
except Exception:
    pass
info["system_python"] = sys.version.split()[0]

# ---- pick the CUDA build ----------------------------------------------------
if CFG["torch_build"] == "auto":
    if info.get("supports_cu130"):
        CFG["torch_build"] = "cu130"
        why = f"driver {info.get('driver')} (>= 580) supports CUDA 13"
    else:
        CFG["torch_build"] = "cu128"
        why = f"driver {info.get('driver')} is older than 580, so cu128 is used"
else:
    why = "forced in Cell 1"
CFG["torch"], CFG["torchvision"] = CFG["builds"][CFG["torch_build"]]
CFG["build_reason"] = why

print("PREFLIGHT")
for k, v in info.items():
    print(f"  {k:<16} {v}")
print(f"\n  selected build   {CFG['torch_build']}  ({why})")
print(f"  torch pin        torch=={CFG['torch']}  torchvision=={CFG['torchvision']}")

warn = []
if info.get("vram_mb") and info["vram_mb"] < 16000:
    warn.append(f"{info['gpu']} has ~{info['vram_mb']}MB VRAM - keep Hires.fix OFF.")
if info.get("compute_cap") and info["compute_cap"].isdigit() and int(info["compute_cap"]) >= 90:
    warn.append("RTX 50-series detected: the repo notes xformers does NOT support RTX 50s.")
if info.get("disk_free_gb", 99) < 12:
    warn.append(f"only {info['disk_free_gb']}GB free on /content - the venv needs ~10GB.")
if warn:
    print("\n  WARNINGS:")
    for w in warn:
        print(f"    ! {w}")
else:
    print("\n  no preflight warnings")

---
## Cell 3 — Google Drive & model tree

Mounts Drive and pre-creates **every** folder `--model-ref` needs. That matters:
`--model-ref` *replaces* the `models/` directory rather than merging into it, so a
missing subfolder produces an error later.

In [ ]:
# %% Cell 3 - GOOGLE DRIVE + MODEL TREE
step(3, "Mount Drive and create the model tree")

from google.colab import drive
drive.mount(DRIVE)

MODEL_REF = f"{ROOT}/models"
OUTPUT = f"{ROOT}/outputs"
SUBS = ['Stable-diffusion', 'VAE', 'Lora', 'ESRGAN', 'ControlNet', 'ControlNetPreprocessor', 'embeddings', 'Adetailer', 'text_encoder']

for p in [ROOT, MODEL_REF, OUTPUT] + [f"{MODEL_REF}/{s}" for s in SUBS]:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

# Keep the resolved layout for the later cells
ENV = {
    "GRADIO_SERVER_NAME": "0.0.0.0",
    "GRADIO_SERVER_PORT": str(CFG["port"]),
    "GRADIO_ANALYTICS_ENABLED": "False",
    "GRADIO_PACKAGE": CFG["gradio_package"],
    "TORCH_INDEX_URL": f"https://download.pytorch.org/whl/{CFG['torch_build']}",
    "TORCH_COMMAND": (
        f"pip install torch=={CFG['torch']}+{CFG['torch_build']} "
        f"torchvision=={CFG['torchvision']}+{CFG['torch_build']} "
        f"--extra-index-url https://download.pytorch.org/whl/{CFG['torch_build']}"
    ),
    "XFORMERS_PACKAGE": (
        f"xformers=={CFG['xformers']} "
        f"--extra-index-url https://download.pytorch.org/whl/{CFG['torch_build']}"
    ),
    # ADetailer + torch>=2.6 needs this
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "true",
    "UV_LINK_MODE": "copy",
}

# Persist so a restarted session can reload the exact same environment
pathlib.Path("/content/forge_env.json").write_text(
    json.dumps({"cfg": CFG, "env": ENV}, indent=2))

print(f"  drive root : {ROOT}")
print(f"  model_ref  : {MODEL_REF}")
print(f"  outputs    : {OUTPUT}")
for s in SUBS:
    print(f"    - {s}")
print("\n  drop your SDXL .safetensors into:", f"{MODEL_REF}/Stable-diffusion")

---
## Cell 4 — Install `uv` and clone the repo

`uv` is **required** - the WebUI's `webui-user.sh` ships `COMMANDLINE_ARGS="--uv"`,
and `--uv` without `uv` installed fails with `uv: command not found` (a real,
reported Neo failure).

In [ ]:
# %% Cell 4 - INSTALL uv + CLONE
step(4, "Install uv and clone the repo")

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"])
quiet([shutil.which("uv") or "uv", "--version"], check=False)

if os.path.isdir(INSTALL):
    print(f"  removing previous clone at {INSTALL}")
    shutil.rmtree(INSTALL)
pathlib.Path(INSTALL).parent.mkdir(parents=True, exist_ok=True)

run(["git", "clone", "--branch", CFG["branch"], "--depth", "1", CFG["repo"], INSTALL])

_, sha = quiet(["git", "rev-parse", "HEAD"], cwd=INSTALL)
CFG["commit"] = sha.strip()
print(f"  branch {CFG['branch']} @ {CFG['commit']}")
print("  tip: pin this SHA so a re-clone can never silently change behaviour:")
print(f"       git checkout {CFG['commit']}")

---
## Cell 5 — Isolated Python 3.13 virtualenv

Colab's system Python is 3.12, but this WebUI requires **3.13**.
`uv venv --python 3.13 --seed` makes `uv` **download a standalone interpreter** -
this is the single most important step, and the reason a plain `pip install` fails.

Every later step uses `INSTALL/venv/bin/python`, so Colab's preinstalled torch,
numpy and gradio can never leak in.

In [ ]:
# %% Cell 5 - VIRTUALENV
step(5, f"Create the Python {CFG['pyver']} venv")

shutil.rmtree(os.path.join(INSTALL, "venv"), ignore_errors=True)
run(["uv", "venv", "venv", "--python", CFG["pyver"], "--seed"], cwd=INSTALL)

VENV_PY = os.path.join(INSTALL, "venv", "bin", "python")
venv_env = {**os.environ, **ENV}

_, pv = quiet([VENV_PY, "-V"])
_, ex = quiet([VENV_PY, "-c", "import sys; print(sys.executable)"])
print(f"  version    {pv.strip()}")
print(f"  executable {ex.strip()}")

major_minor = ".".join(pv.strip().split()[1].split(".")[:2])
if major_minor != CFG["pyver"]:
    raise ForgeError(
        f"venv is Python {major_minor}, expected {CFG['pyver']}", "",
        f"Delete venv/ and re-run. uv must have fetched a {CFG['pyver']} interpreter.")
print(f"  OK - isolated {major_minor} environment ready")

---
## Cell 6 — Install torch, gradio, xformers

Runs `launch.py` once with `--skip-install` **off** so `prepare_environment()` reads
the `TORCH_COMMAND` / `GRADIO_PACKAGE` / `XFORMERS_PACKAGE` env vars we set in Cell 3
and installs exactly the versions this branch expects.

Build matrix detected in Cell 2:
* `cu130` -> torch `2.13.0` / torchvision `0.28.0`
* `cu128` -> torch `2.11.0` / torchvision `0.26.0`

Pinned **before** the first launch so the WebUI never self-installs a wrong version.
**Neo note:** cu128 has no torch 2.13.0 cp313 build, so the cu128 fallback drops to torch 2.11.0. xformers 0.0.35 accepts torch>=2.10, so it still works.

In [ ]:
# %% Cell 6 - FIRST LAUNCH (installs deps)
step(6, "First launch - installs all pinned dependencies")

first_args = ["launch.py"]

first_args += [
    "--no-download-sd-model",
    "--skip-python-version-check",
]

# TORCH_COMMAND / GRADIO_PACKAGE / XFORMERS_PACKAGE all come from ENV (Cell 3)
run([VENV_PY] + first_args, cwd=INSTALL, env=venv_env)

print("\n  dependencies installed")

---
## Cell 7 — ADetailer (optional)

**Do not install Bing-su/adetailer on Neo** (breaks against Neo). Neo also removed the no-op `--disable-safe-unpickle` flag, so the env var is required. mediapipe has no py3.13 wheels - use YOLO detectors.

Set `CFG["install_adetailer"] = False` in Cell 1 to skip this.

In [ ]:
# %% Cell 7 - ADETAILER
if not CFG["install_adetailer"]:
    print("  skipped (CFG['install_adetailer'] is False)")
else:
    step(7, "Install ADetailer")
    ext_dir = os.path.join(INSTALL, "extensions")
    pathlib.Path(ext_dir).mkdir(parents=True, exist_ok=True)
    ad = os.path.join(ext_dir, "adetailer")
    shutil.rmtree(ad, ignore_errors=True)
    run(["git", "clone", "--depth", "1", "https://github.com/abzaloff/aadetailer-neoforge", ad])
    print("  NOTE: re-run the LAUNCH cell (no --skip-install) after adding extensions,")
    print("        otherwise extension requirements are never installed.")

---
## Cell 8 — Verify the environment *before* launching

Runs inside the venv and checks the things that actually break Colab installs:
Python version, torch CUDA availability, device name and compute capability, and
the gradio major version. Prints a report and raises with a specific hint if
something is wrong — cheaper to debug here than 40 cell-lines into a boot.

In [ ]:
# %% Cell 8 - VERIFY
step(8, "Verify the environment")

CHECK = "import sys, json\nimport importlib.metadata as md\ndef v(n):\n    try: return md.version(n)\n    except Exception: return None\ninfo = {'python': sys.version.split()[0]}\nfor n in ('torch','torchvision','gradio','xformers','numpy','pydantic','transformers'):\n    info[n] = v(n)\ntry:\n    import torch\n    info['cuda_available'] = bool(torch.cuda.is_available())\n    info['cuda_runtime'] = torch.version.cuda\n    if torch.cuda.is_available():\n        info['device'] = torch.cuda.get_device_name(0)\n        info['capability'] = '.'.join(map(str, torch.cuda.get_device_capability(0)))\nexcept Exception as e:\n    info['torch_error'] = repr(e)\nprint(json.dumps(info, indent=2))"
rc, _ = run([VENV_PY, "-c", CHECK], cwd=INSTALL, env=venv_env, check=False)

# Re-run capturing JSON only, so we can assert on it
rc, out = quiet([VENV_PY, "-c", CHECK], cwd=INSTALL, env=venv_env, check=False)
try:
    rep = json.loads(out)
except Exception:
    raise ForgeError("could not parse the verification report", out,
                     "The venv python failed to run. Re-run Cell 5 and Cell 6.")

print(json.dumps(rep, indent=2))

problems = []
if not rep.get("python", "").startswith(CFG["pyver"]):
    problems.append(f"python is {rep.get('python')}, expected {CFG['pyver']}*")
if not rep.get("cuda_available"):
    problems.append("torch.cuda.is_available() is False - the CUDA wheel did not load")
if not (rep.get("gradio") or "").startswith(CFG["gradio_expect"]):
    problems.append(f"gradio is {rep.get('gradio')}, expected {CFG['gradio_expect']}.*")
if CFG["use_xformers"] and not rep.get("xformers"):
    problems.append("xformers is not installed but CFG['use_xformers'] is True")

if problems:
    raise ForgeError("environment verification failed", json.dumps(rep, indent=2),
                     " | ".join(problems))
print("\n  OK - python, torch+CUDA and gradio all match this branch's pins")

---
## Cell 9 — Launch

Flags travel through the **`COMMANDLINE_ARGS` environment variable**, because
`modules/paths_internal.py` does:

```python
commandline_args = os.environ.get("COMMANDLINE_ARGS", "")
sys.argv += shlex.split(commandline_args)
```

Passing flags *positionally* to `webui-user.sh` silently drops them (bare `exec`
does not inherit positional params). There is also **no `--extensions-dir` flag** in
either branch — extensions resolve as `<data-dir>/extensions`, so we symlink.

This cell blocks while the WebUI runs. **Stop it with the square button** on the cell.

In [ ]:
# %% Cell 9 - LAUNCH
step(9, "Launch the WebUI")

args = [
    "--no-download-sd-model",
    "--listen",
    "--port", str(CFG["port"]),
    "--model-ref", MODEL_REF,
    "--api",
] + (["--xformers"] if CFG["use_xformers"] else []) + list(CFG["extra_launch_args"])

launch_env = {**os.environ, **venv_env, "COMMANDLINE_ARGS": " ".join(args)}
CFG["launch_args"] = args

print(f"  COMMANDLINE_ARGS = {' '.join(args)}\n")

LOG = "/content/forge_launch.log"
log_fh = open(LOG, "w", encoding="utf-8", errors="replace")
proc = subprocess.Popen(
    [VENV_PY, "launch.py"], cwd=INSTALL, env=launch_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
tail = collections.deque(maxlen=60)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log_fh.write(line)
        tail.append(line)
finally:
    log_fh.close()

out = "".join(tail)
rc = proc.wait()
if rc != 0:
    raise ForgeError(f"WebUI exited with code {rc}", out,
                     hint_for(out) or f"Full log saved to {LOG}")
print(f"\n  WebUI stopped cleanly. Log: {LOG}")

---
## Cell 10 — Reconnect to the UI

After a Colab disconnect the WebUI usually keeps running in the kernel. Re-expose it:

the port you set in Cell 1 (`CFG["port"]`)

Nothing else in this notebook needs re-running.

In [ ]:
# %% Cell 10 - RECONNECT
_port = CFG["port"]
from google.colab.output import eval_js
eval_js(f"google.colab.kernel.proxyPort({_port})")
print(f"  UI: http://localhost:{_port}")

---
## Flag reference

| Flag                    | Effect                                                 |
|-------------------------|--------------------------------------------------------|
| `--xformers`            | installs the package only; selection is automatic      |
| `--uv-local-cache`      | UV_CACHE_DIR inside the WebUI dir - **best for Colab** |
| `--expandable-segments` | experimental allocator; may prevent OOM                |
| `--tiled-conv2d 256`    | tiled VAE convs - big SD1/SDXL VAE saving, but slower  |
| `--use-ck-attention`    | Comfy-Kitchen attention, overrides xformers            |
| `--fast-fp8`            | use torch._scaled_mm for fp8_e4m3fn UNets              |
| `--nunchaku`            | installs nunchaku - **falls back to torch 2.11.0**     |

Environment variables the WebUI reads
(`prepare_environment()` in `modules/launch_utils.py`):
`TORCH_COMMAND` · `TORCH_INDEX_URL` · `GRADIO_PACKAGE` · `XFORMERS_PACKAGE` ·
`PACKAGING_PACKAGE` · `INSIGHT_PACKAGE` · `REQS_FILE` · `PYTORCH_VERSION` ·
`SAGE_PACKAGE` · `FLASH_PACKAGE` · `TRITION_PACKAGE` · `COMMANDLINE_ARGS` ·
`TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD` · `GRADIO_SERVER_NAME` · `GRADIO_SERVER_PORT` ·
`VENV_DIR` · `PYTHON` · `SKIP_VENV`

---

## Troubleshooting

| Symptom                                          | Cause                                          | Fix                                                      |
|--------------------------------------------------|------------------------------------------------|----------------------------------------------------------|
| `uv: command not found`                          | `--uv` used but uv never installed             | re-run the uv install cell (this is a known Neo failure) |
| `This program is tested with 3.13.12`            | Colab system Python is 3.12                    | re-run the venv cell with `--python 3.13`                |
| `Please update your GPU driver to support cu130` | driver older than CUDA 13                      | set CFG['torch_build']='cu128' (drops to torch 2.11.0)   |
| `AssertionError: Torch not compiled with CUDA`   | CPU/ROCm wheel picked                          | delete venv/, re-run venv + build cells                  |
| `UI blank / tabs broken`                         | gradio != 4.40.0 or missing gradio_rangeslider | check the verify cell output                             |
| `ADetailer errors on Neo`                        | wrong fork / mediapipe detectors               | re-run the ADetailer cell, use YOLO detectors            |

### SDXL settings once the UI is up
* **SD VAE → Automatic** (or None). Never an SD1.x VAE with SDXL.
* Sampler `DPM++ 2M` / Scheduler `Karras`, 25–30 steps, CFG 5–7, **1024×1024**.
* **Hires.fix OFF** on a T4 (15 GB). Keep the text encoder on CPU if VRAM is tight.
* First generation is slow (model load + CUDA graph warmup); that is normal.